In [8]:
import logging
import pathlib
from argparse import ArgumentParser, RawTextHelpFormatter
from dataclasses import dataclass
from functools import partial
from typing import Callable

import torch
import torchaudio
import sys
sys.path.append('/om4/group/mcdermott/user/imgriff/projects/pytorch_audio/audio/examples/asr/emformer_rnnt/')
from common import MODEL_TYPE_LIBRISPEECH
from torchaudio.pipelines import EMFORMER_RNNT_BASE_LIBRISPEECH
from torchaudio.pipelines import RNNTBundle
import torchaudio.transforms as T

sys.path.append('/om4/group/mcdermott/user/imgriff/projects/End-to-end-ASR-Pytorch/')
from corpus.single_word import SingleWordDataset as Dataset
from src.text import load_text_encoder


### Pytorch demo format 

In [2]:
@dataclass
class Config:
    dataset: Callable
    bundle: RNNTBundle

In [3]:

_CONFIGS = {
    MODEL_TYPE_LIBRISPEECH: Config(
        partial(torchaudio.datasets.LIBRISPEECH, url="test-clean"),
        EMFORMER_RNNT_BASE_LIBRISPEECH)
}

### Set params 

In [4]:
model_type = 'librispeech'

path = '/om2/user/imgriff/projects/cocktail_party/datasets/GigaSpeech_top_200_words/' # Path to raw LibriSpeech dataset
dev_split = ['GigaSpeech_top_200_words']  # Name of data splits to be used as validation set

mode = 'word'                       # 'character'/'word'/'subword'
vocab_file = '/om4/group/mcdermott/user/imgriff/projects/End-to-end-ASR-Pytorch/tests/sample_data/gigaspeech_words-10k.vocab'


### Load model

In [15]:
# dataset 
dataset = Dataset(path, dev_split, None, 1)

# model 
bundle = _CONFIGS[model_type].bundle
decoder = bundle.get_decoder().cuda()
token_processor = bundle.get_token_processor()
feature_extractor = bundle.get_feature_extractor().cuda()
streaming_feature_extractor = bundle.get_streaming_feature_extractor().cuda()
hop_length = bundle.hop_length
num_samples_segment = bundle.segment_length * hop_length
num_samples_segment_right_context = num_samples_segment + bundle.right_context_length * hop_length

Using custom data configuration default-d77d8d0e2c068ac8
Reusing dataset csv (/home/imgriff/.cache/huggingface/datasets/csv/default-d77d8d0e2c068ac8/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e)


#### Single words recorded at 44.1kHz - resample to 16kHz on the fly

In [17]:
resampler = T.Resample(44100, 16000, lowpass_filter_width=64,
                   rolloff=0.9475937167399596, 
                   resampling_method="kaiser_window",beta=14.769656459379492, dtype=torch.float32)

def load_audio(example):
    wav, sr = torchaudio.load(example['wav_path'])
    wav = resampler(wav)
    example['audio'] = wav
    example['sampling_rate'] = 16000
    return example


Unpack huggingface dataset from custom dataset object for convenience 

In [18]:
updated_dataset = dataset.dataset.map(load_audio)

0ex [00:00, ?ex/s]

#### Run Single-Word evaluation

In [29]:
out_file = '../result/emformer_single_word.csv'

with open(out_file,'w',encoding='UTF-8') as f:
                f.write('idx\tbeam\thyp\ttruth\n')

for sample in updated_dataset:
    truth = sample["word"].lower()

    waveform = torch.FloatTensor(sample['audio'][0]).cuda()
    # Streaming decode.
#     state, hypothesis = None, None
#     for idx in range(0, len(waveform), num_samples_segment):
#         segment = waveform[idx : idx + num_samples_segment_right_context]
#         segment = torch.nn.functional.pad(segment, (0, num_samples_segment_right_context - len(segment)))
#         with torch.no_grad():
#             features, length = streaming_feature_extractor(segment)
#             hypos, state = decoder.infer(features, length, 10, state=state, hypothesis=hypothesis)
#         hypothesis = hypos[0]
#         transcript = token_processor(hypothesis.tokens, lstrip=False)
#         print(transcript, end="", flush=True)
#     print()

    # Non-streaming decode.
    with torch.no_grad():
        features, length = feature_extractor(waveform)
        hypos = decoder(features, length, 10)
 
            
#     print([token_processor(hypos[ix].tokens) for ix in range(10)])
#     print(f"truth: {sample[1]}")
#     print()

    hypos = [token_processor(hypos[ix].tokens) for ix in range(10)]
    with open(out_file, 'a', encoding='UTF-8') as f:
        for b, hyp in enumerate(hypos):
            f.write('\t'.join([truth, str(b), hyp, truth])+'\n')


In [30]:
import pandas as pd

In [31]:
results = pd.read_csv(out_file, sep='\t')

In [32]:
results.head(50)

,idx,beam,hyp,truth
0,the,0,the,the
1,the,1,the the,the
2,the,2,t,the
3,the,3,t the,the
4,the,4,ta,the
5,the,5,ca,the
6,the,6,vidocq,the
7,the,7,ye the,the
8,the,8,neither,the
9,the,9,either,the


In [42]:
top_match_guesses = results[(results['hyp'] == results['truth']) & (results['beam']==0)]
percent_right = 100 * len(top_match_guesses) / len(results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses

167 matched guesses; 83.5% correct


,idx,beam,hyp,truth
0,the,0,the,the
10,and,0,and,and
30,of,0,of,of
40,a,0,a,a
50,that,0,that,that
...,...,...,...,...
1930,while,0,while,while
1940,big,0,big,big
1950,twenty,0,twenty,twenty
1960,each,0,each,each
